# Modelo-07 — Treinamento do CatBoostClassifier

## Objetivo

Este notebook treina e avalia um `CatBoostClassifier` para prever se um sistema PSD apresentará ao menos uma falha confirmada entre `D+1` e `D+7`.

O treinamento utilizará as bases temporais produzidas no notebook de processamento:

- **treino:** ajuste dos parâmetros internos do modelo;
- **validação:** acompanhamento da generalização, seleção de iterações e ajuste do limiar de decisão;
- **teste:** avaliação final, utilizada somente depois que o modelo e o limiar estiverem definidos.

As principais métricas serão `PR-AUC`, `ROC-AUC`, `recall`, `precision`, `F1` e matriz de confusão. Como o objetivo é antecipar falhas, o limiar não será escolhido apenas pela acurácia: será necessário equilibrar falhas detectadas e quantidade de alertas falsos.

> O conjunto de teste não será utilizado para ajustar hiperparâmetros, selecionar features ou escolher o limiar de classificação.

In [3]:
from pathlib import Path
import json

import catboost
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from catboost import CatBoostClassifier, Pool
from IPython.display import display
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda valor: f"{valor:,.4f}")

print(f"CatBoost: {catboost.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"Scikit-learn: {sklearn.__version__}")

CatBoost: 1.2.10
Pandas: 3.0.5
Scikit-learn: 1.9.0


In [5]:
DIRETORIO_DADOS = Path("../../data/processados/modelo-07")

with (DIRETORIO_DADOS / "configuracao_modelagem.json").open(encoding="utf-8") as arquivo:
    configuracao = json.load(arquivo)

TARGET = configuracao["target"]
features_categoricas = configuracao["features_categoricas"]
features_numericas = configuracao["features_numericas"]
features_modelo = configuracao["features_modelo"]
RANDOM_SEED = configuracao["random_seed"]

def carregar_base(nome_conjunto):
    df = pd.read_csv(
        DIRETORIO_DADOS / f"base_{nome_conjunto}.csv",
        parse_dates=["data_referencia"],
    )

    for coluna in features_categoricas:
        df[coluna] = df[coluna].astype(str)

    return df

base_treino = carregar_base("treino")
base_validacao = carregar_base("validacao")
base_teste = carregar_base("teste")

auditoria_bases = pd.DataFrame([
    {
        "conjunto": nome,
        "linhas": len(df),
        "data_inicial": df["data_referencia"].min(),
        "data_final": df["data_referencia"].max(),
        "taxa_positivos": df[TARGET].mean() * 100,
    }
    for nome, df in {
        "treino": base_treino,
        "validacao": base_validacao,
        "teste": base_teste,
    }.items()
])

display(auditoria_bases)

,conjunto,linhas,data_inicial,data_final,taxa_positivos
0,treino,16016,2023-03-28,2025-03-24,45.3546
1,validacao,3872,2025-04-01,2025-09-23,52.1436
2,teste,3784,2025-10-01,2026-03-21,51.7178


In [6]:
def separar_features_alvo(df):
    X = df[features_modelo].copy()
    y = df[TARGET].astype("int8").copy()
    return X, y

X_treino, y_treino = separar_features_alvo(base_treino)
X_validacao, y_validacao = separar_features_alvo(base_validacao)
X_teste, y_teste = separar_features_alvo(base_teste)

for X in [X_treino, X_validacao, X_teste]:
    assert X.columns.tolist() == features_modelo
    assert not X[features_categoricas].isna().any().any()

pool_treino = Pool(
    data=X_treino,
    label=y_treino,
    cat_features=features_categoricas,
)
pool_validacao = Pool(
    data=X_validacao,
    label=y_validacao,
    cat_features=features_categoricas,
)
pool_teste = Pool(
    data=X_teste,
    label=y_teste,
    cat_features=features_categoricas,
)

print(f"X_treino: {X_treino.shape} | y_treino: {y_treino.shape}")
print(f"X_validacao: {X_validacao.shape} | y_validacao: {y_validacao.shape}")
print(f"X_teste: {X_teste.shape} | y_teste: {y_teste.shape}")
print(f"Features categóricas: {features_categoricas}")

X_treino: (16016, 21) | y_treino: (16016,)
X_validacao: (3872, 21) | y_validacao: (3872,)
X_teste: (3784, 21) | y_teste: (3784,)
Features categóricas: ['id_psd', 'estacao_padrao', 'plataforma_padrao', 'dia_semana']


In [7]:
parametros_baseline = {
    "iterations": 1_000,
    "learning_rate": 0.05,
    "depth": 6,
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "l2_leaf_reg": 3.0,
    "random_strength": 1.0,
    "random_seed": RANDOM_SEED,
    "allow_writing_files": False,
    "verbose": 100,
}

modelo_baseline = CatBoostClassifier(**parametros_baseline)

display(pd.DataFrame(
    parametros_baseline.items(),
    columns=["parametro", "valor"],
))

,parametro,valor
0,iterations,1000
1,learning_rate,0.0500
2,depth,6
3,loss_function,Logloss
4,eval_metric,AUC
5,l2_leaf_reg,3.0000
6,random_strength,1.0000
7,random_seed,42
8,allow_writing_files,False
9,verbose,100


In [8]:
modelo_baseline.fit(
    pool_treino,
    eval_set=pool_validacao,
    use_best_model=True,
    early_stopping_rounds=100,
)

melhor_iteracao = modelo_baseline.get_best_iteration()
melhores_resultados = modelo_baseline.get_best_score()

print(f"Melhor iteração: {melhor_iteracao}")
print(f"Melhor AUC de validação: {melhores_resultados['validation']['AUC']:.4f}")
print(f"Logloss de validação nessa seleção: {melhores_resultados['validation']['Logloss']:.4f}")

0:	test: 0.5772160	best: 0.5772160 (0)	total: 204ms	remaining: 3m 24s
100:	test: 0.5852531	best: 0.5900638 (52)	total: 4.28s	remaining: 38.1s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.5900638484
bestIteration = 52

Shrink model to first 53 iterations.
Melhor iteração: 52
Melhor AUC de validação: 0.5901
Logloss de validação nessa seleção: 0.6780
